In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum
from datetime import datetime

In [2]:
import os
os.environ["HADOOP_HOME"] = "C:\\hadoop"
os.environ["PATH"] = os.environ["PATH"] + ";C:\\hadoop\\bin"

spark = SparkSession.builder \
    .appName("clean") \
    .getOrCreate()

In [3]:
customer = spark.read.parquet('../spark/rawdata/20260330_160319_customer.parquet')
product = spark.read.parquet('../spark/rawdata/20260330_160322_product.parquet')
transaction = spark.read.parquet('../spark/rawdata/20260331_132527_transaction.parquet')

# Customer_cleaning

In [4]:
customer.show()

+----------+--------------+-----------------+
|CustomerNo|       Country|             Name|
+----------+--------------+-----------------+
|   17490.0|United Kingdom|     Sara Griffin|
|   13069.0|United Kingdom|     Michael Holt|
|   12433.0|        Norway|   Kelli Sandoval|
|   13426.0|United Kingdom|    Dalton Graves|
|   17364.0|United Kingdom|   Michelle James|
|   14441.0|United Kingdom|      Thomas Hull|
|   16446.0|United Kingdom|       John Burns|
|   17389.0|United Kingdom|    Victoria King|
|   17001.0|United Kingdom|     Kayla Stuart|
|   15694.0|United Kingdom|     Mark Andrews|
|   17428.0|United Kingdom|    Tammy Hammond|
|   16954.0|United Kingdom|   Zachary Knight|
|   15492.0|United Kingdom|    April Simmons|
|   12423.0|       Belgium|       April Reed|
|   12518.0|       Germany|    Molly Aguirre|
|   14051.0|United Kingdom|Antonio Rodriguez|
|   16558.0|United Kingdom|    William Jones|
|   17497.0|United Kingdom|Christina Charles|
|   14498.0|United Kingdom|    Hea

In [5]:
customer.select([sum(col(column).isNull().cast("int")).alias(column) for column in customer.columns]).show()

+----------+-------+----+
|CustomerNo|Country|Name|
+----------+-------+----+
|         1|      0|   0|
+----------+-------+----+



In [6]:
customer.where(customer.CustomerNo.isNull()).show()

+----------+--------------+------------+
|CustomerNo|       Country|        Name|
+----------+--------------+------------+
|      NULL|United Kingdom|Allen Morgan|
+----------+--------------+------------+



In [7]:
clean_customer = customer.dropna()
clean_customer.select([sum(col(column).isNull().cast("int")).alias(column) for column in clean_customer.columns]).show()

+----------+-------+----+
|CustomerNo|Country|Name|
+----------+-------+----+
|         0|      0|   0|
+----------+-------+----+



In [8]:
clean_customer.select([sum(col(column).isNull().cast("int")).alias(column) for column in clean_customer.columns]).show()

+----------+-------+----+
|CustomerNo|Country|Name|
+----------+-------+----+
|         0|      0|   0|
+----------+-------+----+



In [9]:
# clean_customer.toPandas().to_parquet(
#     f"../spark/cleandata/{datetime.now().strftime('%Y%m%d')}_customer.parquet",
#     index=False
# )

# Product Cleaning

In [10]:
product.show()

+---------+--------------------+
|ProductNo|         ProductName|
+---------+--------------------+
|    22485|Set Of 2 Wooden M...|
|    22596|Christmas Star Wi...|
|    23235|Storage Tin Vinta...|
|    23272|Tree T-Light Hold...|
|    23239|Set Of 4 Knick Kn...|
|    21705|Bag 500g Swirly M...|
|    22118|Joy Wooden Block ...|
|    22119|Peace Wooden Bloc...|
|    22217|T-Light Holder Ha...|
|    22216|T-Light Holder Wh...|
|    22380|   Toy Tidy Spaceboy|
|    22442|Grow Your Own Flo...|
|    22664|Toy Tidy Dolly Gi...|
|    22721|Set Of 3 Cake Tin...|
|    22723|Set Of 6 Herb Tin...|
|    22785|Squarecushion Cov...|
|    22955|36 Foil Star Cake...|
|    23141|Triple Wire Hook ...|
|    22956|36 Foil Heart Cak...|
|    22581|Wood Stocking Chr...|
+---------+--------------------+
only showing top 20 rows


In [11]:
product.summary().show()

+-------+------------------+--------------------+
|summary|         ProductNo|         ProductName|
+-------+------------------+--------------------+
|  count|              3768|                3768|
|   mean|32685.657326130993|                NULL|
| stddev| 22985.07680799358|                NULL|
|    min|             10002|10 Colour Spacebo...|
|    25%|           21793.0|                NULL|
|    50%|           22678.0|                NULL|
|    75%|           23463.0|                NULL|
|    max|            90214Z|Zinc Wire Sweethe...|
+-------+------------------+--------------------+



In [12]:
# product.toPandas().to_parquet(
#     f"../spark/cleandata/{datetime.now().strftime('%Y%m%d')}_product.parquet",
#     index=False
# )

# Transaction Cleaning

In [13]:
transaction.select([sum(col(column).isNull().cast("int")).alias(column) for column in transaction.columns]).show()

+-------------+---------+---------+-----+--------+----------+
|TransactionNo|Date_time|ProductNo|Price|Quantity|CustomerNo|
+-------------+---------+---------+-----+--------+----------+
|            0|        0|        0|    0|       0|        55|
+-------------+---------+---------+-----+--------+----------+



In [14]:
transaction.where(transaction.CustomerNo.isNull()).show()

+-------------+-------------------+---------+-----+--------+----------+
|TransactionNo|          Date_time|ProductNo|Price|Quantity|CustomerNo|
+-------------+-------------------+---------+-----+--------+----------+
|      C581406|2024-05-09 00:00:00|   46000M| 6.19|    -240|      NULL|
|      C581406|2024-05-09 00:00:00|   46000S| 6.19|    -300|      NULL|
|      C575153|2024-04-09 00:00:00|    22947|44.25|      -1|      NULL|
|      C574288|2024-04-04 00:00:00|    22178|25.37|      -1|      NULL|
|      C573180|2024-03-29 00:00:00|    23048| 14.5|      -1|      NULL|
|      C569495|2024-03-05 00:00:00|    21843|21.47|      -1|      NULL|
|      C567518|2024-02-20 00:00:00|    22846|27.62|      -1|      NULL|
|      C567518|2024-02-20 00:00:00|    21871|11.94|     -12|      NULL|
|      C563015|2024-01-11 00:00:00|   46000M|10.25|    -160|      NULL|
|      C563015|2024-01-11 00:00:00|   46000S|10.25|    -220|      NULL|
|      C562617|2024-01-08 00:00:00|    23243|15.32|      -2|    

In [15]:
transaction_drop = transaction.dropna()
transaction_drop.select([sum(col(column).isNull().cast('int')).alias(column) for column in transaction_drop.columns]).show()

+-------------+---------+---------+-----+--------+----------+
|TransactionNo|Date_time|ProductNo|Price|Quantity|CustomerNo|
+-------------+---------+---------+-----+--------+----------+
|            0|        0|        0|    0|       0|         0|
+-------------+---------+---------+-----+--------+----------+



In [16]:
transaction_outlier = transaction_drop.where(transaction_drop.Price >= 0) and transaction_drop.where(transaction_drop.Quantity > 0)

In [17]:
transaction_outlier.summary().show()

+-------+------------------+------------------+------------------+------------------+------------------+
|summary|     TransactionNo|         ProductNo|             Price|          Quantity|        CustomerNo|
+-------+------------------+------------------+------------------+------------------+------------------+
|  count|            527764|            527764|            527764|            527764|            527764|
|   mean| 559978.9329113013|27495.740468255117|12.629640123247487|10.594678682138229|15231.626732782077|
| stddev|13434.279595094802|16616.851795150527|7.9332242286321915|156.78679493942042| 1716.522181772706|
|    min|            536365|             10002|              5.13|                 1|           12004.0|
|    25%|          547898.0|           21928.0|             10.99|                 1|           13813.0|
|    50%|          560701.0|           22565.0|             11.94|                 3|           15159.0|
|    75%|          571848.0|           23160.0|        

In [18]:
clean_transaction = transaction_outlier.withColumn("Total_sales", col('Price') * col('Quantity'))
clean_transaction.show()

+-------------+-------------------+---------+-----+--------+----------+------------------+
|TransactionNo|          Date_time|ProductNo|Price|Quantity|CustomerNo|       Total_sales|
+-------------+-------------------+---------+-----+--------+----------+------------------+
|       581482|2024-05-10 00:00:00|    22485|21.47|      12|   17490.0|            257.64|
|       581475|2024-05-10 00:00:00|    22596|10.65|      36|   13069.0|383.40000000000003|
|       581475|2024-05-10 00:00:00|    23235|11.53|      12|   13069.0|138.35999999999999|
|       581475|2024-05-10 00:00:00|    23272|10.65|      12|   13069.0|127.80000000000001|
|       581475|2024-05-10 00:00:00|    23239|11.94|       6|   13069.0|             71.64|
|       581475|2024-05-10 00:00:00|    21705|10.65|      24|   13069.0|255.60000000000002|
|       581475|2024-05-10 00:00:00|    22118|11.53|      18|   13069.0|            207.54|
|       581475|2024-05-10 00:00:00|    22119|12.25|      12|   13069.0|             147.0|

In [19]:
clean_transaction = clean_transaction.withColumn("CustomerNo", col('CustomerNo').cast('int').cast('string'))

In [20]:
clean_transaction.show()

+-------------+-------------------+---------+-----+--------+----------+------------------+
|TransactionNo|          Date_time|ProductNo|Price|Quantity|CustomerNo|       Total_sales|
+-------------+-------------------+---------+-----+--------+----------+------------------+
|       581482|2024-05-10 00:00:00|    22485|21.47|      12|     17490|            257.64|
|       581475|2024-05-10 00:00:00|    22596|10.65|      36|     13069|383.40000000000003|
|       581475|2024-05-10 00:00:00|    23235|11.53|      12|     13069|138.35999999999999|
|       581475|2024-05-10 00:00:00|    23272|10.65|      12|     13069|127.80000000000001|
|       581475|2024-05-10 00:00:00|    23239|11.94|       6|     13069|             71.64|
|       581475|2024-05-10 00:00:00|    21705|10.65|      24|     13069|255.60000000000002|
|       581475|2024-05-10 00:00:00|    22118|11.53|      18|     13069|            207.54|
|       581475|2024-05-10 00:00:00|    22119|12.25|      12|     13069|             147.0|

In [21]:
pp = clean_transaction.toPandas()
pp

,TransactionNo,Date_time,ProductNo,Price,Quantity,CustomerNo,Total_sales
0,581482,2024-05-10,22485,21.47,12,17490,257.64
1,581475,2024-05-10,22596,10.65,36,13069,383.40
2,581475,2024-05-10,23235,11.53,12,13069,138.36
3,581475,2024-05-10,23272,10.65,12,13069,127.80
4,581475,2024-05-10,23239,11.94,6,13069,71.64
...,...,...,...,...,...,...,...
527759,536585,2023-05-03,37449,20.45,2,17460,40.90
527760,536590,2023-05-03,22776,20.45,1,13065,20.45
527761,536590,2023-05-03,22622,20.45,2,13065,40.90
527762,536591,2023-05-03,37449,20.45,1,14606,20.45


In [22]:
# clean_transaction.toPandas().to_parquet(
#     f"../spark/cleandata/{datetime.now().strftime('%Y%m%d')}_transaction.parquet",
#     index=False
# )

# Join Data

In [23]:
clean_customer.createOrReplaceTempView('customer')
clean_transaction.createOrReplaceTempView('transaction')
product.createOrReplaceTempView('product')

In [42]:
product.show()

+---------+--------------------+
|ProductNo|         ProductName|
+---------+--------------------+
|    22485|Set Of 2 Wooden M...|
|    22596|Christmas Star Wi...|
|    23235|Storage Tin Vinta...|
|    23272|Tree T-Light Hold...|
|    23239|Set Of 4 Knick Kn...|
|    21705|Bag 500g Swirly M...|
|    22118|Joy Wooden Block ...|
|    22119|Peace Wooden Bloc...|
|    22217|T-Light Holder Ha...|
|    22216|T-Light Holder Wh...|
|    22380|   Toy Tidy Spaceboy|
|    22442|Grow Your Own Flo...|
|    22664|Toy Tidy Dolly Gi...|
|    22721|Set Of 3 Cake Tin...|
|    22723|Set Of 6 Herb Tin...|
|    22785|Squarecushion Cov...|
|    22955|36 Foil Star Cake...|
|    23141|Triple Wire Hook ...|
|    22956|36 Foil Heart Cak...|
|    22581|Wood Stocking Chr...|
+---------+--------------------+
only showing top 20 rows


In [79]:
query = """
with data1 as(
    select 
        TransactionNo,
        Date_time,
        ProductNo,
        Price,
        Quantity,
        t.CustomerNo,
        Total_sales,
        Country,
        Name
    from transaction t
    left join customer c
    on t.CustomerNo = c.CustomerNo
    )

select 
    d.TransactionNo,
    d.Date_time,
    d.ProductNo,
    d.Price,
    d.Quantity,
    d.CustomerNo,
    d.Total_sales,
    d.Country,
    d.Name,
    p.ProductName
from data1 d
left join product p
on d.ProductNo = p.ProductNo;"""

In [80]:
data = spark.sql(query)
data.show()

+-------------+-------------------+---------+-----+--------+----------+------------------+--------------+------------+--------------------+
|TransactionNo|          Date_time|ProductNo|Price|Quantity|CustomerNo|       Total_sales|       Country|        Name|         ProductName|
+-------------+-------------------+---------+-----+--------+----------+------------------+--------------+------------+--------------------+
|       581482|2024-05-10 00:00:00|    22485|21.47|      12|     17490|            257.64|United Kingdom|Sara Griffin|Set Of 2 Wooden M...|
|       581475|2024-05-10 00:00:00|    22596|10.65|      36|     13069|383.40000000000003|United Kingdom|Michael Holt|Christmas Star Wi...|
|       581475|2024-05-10 00:00:00|    23235|11.53|      12|     13069|138.35999999999999|United Kingdom|Michael Holt|Storage Tin Vinta...|
|       581475|2024-05-10 00:00:00|    23272|10.65|      12|     13069|127.80000000000001|United Kingdom|Michael Holt|Tree T-Light Hold...|
|       581475|2024-

In [82]:
data.select([sum(col(c).isNull().cast('int')).alias(c) for c in data.columns]).show()

+-------------+---------+---------+-----+--------+----------+-----------+-------+----+-----------+
|TransactionNo|Date_time|ProductNo|Price|Quantity|CustomerNo|Total_sales|Country|Name|ProductName|
+-------------+---------+---------+-----+--------+----------+-----------+-------+----+-----------+
|            0|        0|        0|    0|       0|         0|          0|      0|   0|          0|
+-------------+---------+---------+-----+--------+----------+-----------+-------+----+-----------+



In [88]:
data = data.withColumn(
    "Date_time",
    col("Date_time").cast("date")
)

In [89]:
data.printSchema()

root
 |-- TransactionNo: string (nullable = true)
 |-- Date_time: date (nullable = true)
 |-- ProductNo: string (nullable = true)
 |-- Price: double (nullable = true)
 |-- Quantity: long (nullable = true)
 |-- CustomerNo: string (nullable = true)
 |-- Total_sales: double (nullable = true)
 |-- Country: string (nullable = true)
 |-- Name: string (nullable = true)
 |-- ProductName: string (nullable = true)



In [90]:
data.toPandas().to_parquet(
    f"../spark/cleandata/{datetime.now().strftime('%Y%m%d')}_full_transaction.parquet",
    index=False
)